In [1]:
!pip install azure-storage-blob
from kaggle_secrets import UserSecretsClient
from azure.storage.blob import BlobServiceClient, BlobClient, ContainerClient
import numpy as np
import pandas as pd
import random
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer, BertModel
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Embedding, Flatten, Concatenate, Dropout
user_secrets = UserSecretsClient()
connection_string = user_secrets.get_secret("AZURE_CONNECTION_STRING")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 408.6/408.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 14.0 MB/s eta 0:00:00


In [2]:
container_name = "mind-data"
blob_name = "transformed-data/behaviors/behaviors.parquet"
download_file_path = "/kaggle/working/behaviors.parquet"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_client = blob_service_client.get_container_client(container_name)

blob_client = container_client.get_blob_client(blob_name)

with open(download_file_path, "wb") as file:
    download_stream = blob_client.download_blob()
    file.write(download_stream.readall())

print(f"Parquet file downloaded to {download_file_path}")

Parquet file downloaded to /kaggle/working/behaviors.parquet


In [3]:
container_name = "mind-data"
blob_name = "transformed-data/news/news.parquet"
download_file_path = "/kaggle/working/news.parquet"

blob_service_client = BlobServiceClient.from_connection_string(connection_string)

container_client = blob_service_client.get_container_client(container_name)

blob_client = container_client.get_blob_client(blob_name)

with open(download_file_path, "wb") as file:
    download_stream = blob_client.download_blob()
    file.write(download_stream.readall())

print(f"Parquet file downloaded to {download_file_path}")

Parquet file downloaded to /kaggle/working/news.parquet


In [4]:
news = pd.read_parquet("/kaggle/working/news.parquet")
news.head()

,News ID,Category,SubCategory,Title,Abstract,URL,Title Entities,Abstract Entities,Content
0,N58547,lifestyle,lifestyle horoscope,zodiac friendships that are creative power houses,seeking out a zodiac pair that compliments you...,https://assets.msn.com/labs/mind/AAHcfBo.html,[],[],lifestylelifestyle horoscopezodiac friendships...
1,N4505,lifestyle,lifestyle pets animals,50 incredible photos of animals in the wild,nothing beats taking a short break in your day...,https://assets.msn.com/labs/mind/AAJl7Vo.html,[],[],lifestylelifestyle pets animals50 incredible p...
2,N65132,food and drink,food news,this is the most hated halloween candy in america,"halloween is great, and candy is great, but no...",https://assets.msn.com/labs/mind/AAJl7a0.html,"[{""Label"": ""Halloween"", ""Type"": ""H"", ""Wikidata...","[{""Label"": ""Halloween"", ""Type"": ""H"", ""Wikidata...",food drinkfood newsthis hated halloween candy ...
3,N24566,autos,autos enthusiasts,2020 winnebago solis camper van is made for th...,live life off the grid in a class b van down b...,https://assets.msn.com/labs/mind/AAIfJoX.html,[],[],autosautos enthusiasts2020 winnebago solis cam...
4,N64365,sports,golf,"with poy locked up, j.y. ko competes in swingi...",jin young ko has already secured the rolex pla...,https://assets.msn.com/labs/mind/AAJwjO4.html,[],"[{""Label"": ""Nelly Korda"", ""Type"": ""P"", ""Wikida...",sportsgolfwith poy locked jy ko competes swing...


In [5]:
behaviors = pd.read_parquet("/kaggle/working/behaviors.parquet")
behaviors.head()

,Impression ID,User ID,Timestamp,History,Impressions
0,88,U69950,2019-11-14 10:57:37,N10347 N11282 N12194 N16304 N18094 N18360 N188...,N10960-0 N61296-0 N6578-0 N52554-0 N62318-0 N4...
1,98,U47761,2019-11-13 06:07:16,N11821 N16384 N19079 N22479 N27642 N28614 N312...,N9734-0 N25949-0 N47061-0 N14726-0 N59272-0 N3...
2,284,U10932,2019-11-12 13:47:18,N17589 N29802 N35022 N3560 N36270 N49481 N5047...,N20015-0 N45389-0 N62386-0 N22339-0 N21428-0 N...
3,408,U13674,2019-11-13 15:18:02,N12349 N17109 N25739 N28030 N29068 N31801 N365...,N36638-0 N56214-1 N45509-0 N26376-0 N11769-0 N...
4,770,U38627,2019-11-14 04:48:10,N21136 N2309 N26026 N306 N36530,N6816-0 N4404-0 N40559-0 N42457-0 N30089-0 N80...


In [6]:
def process_impressions(impression_str):
    if isinstance(impression_str, str):
        impressions = impression_str.split()
        return [(imp.split('-')[0], int(imp.split('-')[1])) for imp in impressions]
    else:
        return []

behaviors['Impressions'] = behaviors['Impressions'].apply(process_impressions)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [8]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased').to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [9]:
def get_bert_embedding(text):
    if isinstance(text, str) and text:
        inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = bert_model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1).squeeze()
        return embeddings.cpu().numpy()
    else:
        return [0] * 768

In [10]:
news['Combined_embedding'] = news['Content'].apply(lambda x: get_bert_embedding(x))

In [11]:
news_embeddings_dict = dict(zip(news['News ID'], news['Combined_embedding']))

In [12]:
def map_impressions_to_embeddings(impressions):
    if isinstance(impressions, list):
        return [(news_embeddings_dict.get(news_id, [0]*768), click) for news_id, click in impressions if news_id in news_embeddings_dict]
    else:
        return []

In [13]:
behaviors['Impressions_embeddings'] = behaviors['Impressions'].apply(map_impressions_to_embeddings)

In [14]:
user_item_interactions = []
for index, row in behaviors.iterrows():
    for impression in row['Impressions']:
        news_id, click_label = impression
        user_item_interactions.append([row['User ID'], news_id, click_label])

interaction_df = pd.DataFrame(user_item_interactions, columns=['User ID', 'News ID', 'Click Label'])

user_id_mapping = {id: idx for idx, id in enumerate(interaction_df['User ID'].unique())}
news_id_mapping = {id: idx for idx, id in enumerate(interaction_df['News ID'].unique())}

interaction_df['User Index'] = interaction_df['User ID'].map(user_id_mapping)
interaction_df['News Index'] = interaction_df['News ID'].map(news_id_mapping)

train_df, test_df = train_test_split(interaction_df, test_size=0.2, random_state=42)

In [15]:
train_user_input = train_df['User Index'].values
train_news_input = train_df['News Index'].values
train_labels = train_df['Click Label'].values

num_users = len(user_id_mapping)
num_items = len(news_id_mapping)
embedding_size = 64

user_input = Input(shape=(1,), name='user_input')
news_input = Input(shape=(1,), name='news_input')

user_embedding = Embedding(input_dim=num_users, output_dim=embedding_size)(user_input)
news_embedding = Embedding(input_dim=num_items, output_dim=embedding_size)(news_input)

user_vecs = Flatten()(user_embedding)
news_vecs = Flatten()(news_embedding)

input_vecs = Concatenate()([user_vecs, news_vecs])

x = Dense(128, activation='relu')(input_vecs)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)
ncf_model = Model(inputs=[user_input, news_input], outputs=output)
ncf_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ncf_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ user_input (InputLayer)   │ (None, 1)              │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ news_input (InputLayer)   │ (None, 1)              │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding (Embedding)     │ (None, 1, 64)          │      3,200,000 │ user_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_1 (Embedding)   │ (None, 1, 64)          │      1,298,432 │ news_input[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten (Flatten)         │ (None, 64)             │              0 │ embedding[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_1 (Flatten)       │ (None, 64)             │              0 │ embedding_1[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 128)            │              0 │ flatten[0][0],         │
│                           │                        │                │ flatten_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 128)            │         16,512 │ concatenate[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 128)            │              0 │ dense[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 64)             │          8,256 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_1 (Dropout)       │ (None, 64)             │              0 │ dense_1[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 1)              │             65 │ dropout_1[0][0]        │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 4,523,265 (17.25 MB)

 Trainable params: 4,523,265 (17.25 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
ncf_model.fit([train_user_input, train_news_input], train_labels, epochs=10, batch_size=256, validation_split=0.1)

Epoch 1/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step - accuracy: 0.9585 - loss: 0.1705 - val_accuracy: 0.9594 - val_loss: 0.1562
Epoch 2/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9597 - loss: 0.1519 - val_accuracy: 0.9595 - val_loss: 0.1561
Epoch 3/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9599 - loss: 0.1498 - val_accuracy: 0.9594 - val_loss: 0.1564
Epoch 4/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9603 - loss: 0.1476 - val_accuracy: 0.9592 - val_loss: 0.1573
Epoch 5/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9604 - loss: 0.1466 - val_accuracy: 0.9592 - val_loss: 0.1572
Epoch 6/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9607 - loss: 0.1446 - val_accuracy: 0.9588 - val_loss: 0.1582
Epoch 7/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 0.9610 - loss: 0.1434 - val_accuracy: 0.9590 - val_loss: 0.1586
Epoch 8/10
16435/16435 ━━━━━━━━━━━━━━━━━━━━ 35s 2ms/step - accuracy: 

In [17]:
test_user_input = test_df['User Index'].values
test_news_input = test_df['News Index'].values
test_labels = test_df['Click Label'].values

loss, accuracy = ncf_model.evaluate([test_user_input, test_news_input], test_labels)

36522/36522 ━━━━━━━━━━━━━━━━━━━━ 47s 1ms/step - accuracy: 0.9588 - loss: 0.1603


In [18]:
def generate_recommendations(user_identifier=None, num_recommendations=10):
    if user_identifier is None:
        user_identifier = random.choice(list(user_id_mapping.keys()))
    
    mapped_user_idx = user_id_mapping.get(user_identifier)
    
    if mapped_user_idx is None:
        raise ValueError(f"User ID {user_identifier} does not exist in the mapping.")

    unseen_news_ids = [
        news for news in news_id_mapping.keys() 
        if news not in behaviors[behaviors['User ID'] == user_identifier]['Impressions']
                    .apply(lambda imp: imp[0]).tolist()
    ]
    unseen_indices = np.array([news_id_mapping[news] for news in unseen_news_ids])

    predicted_scores = ncf_model.predict([np.full(len(unseen_indices), mapped_user_idx), unseen_indices])
    results = pd.DataFrame({'News ID': unseen_news_ids, 'Score': predicted_scores.flatten()})
    return user_identifier, results.sort_values(by='Score', ascending=False).head(num_recommendations)

random_user_id, user_recommendations = generate_recommendations()
print(f"Top News Recommendations for User {random_user_id}:")
print(user_recommendations)

634/634 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Top News Recommendations for User U31774:
      News ID     Score
9177   N30631  0.900598
1652   N40629  0.889205
585    N49279  0.880320
2698   N47380  0.761912
14305  N23147  0.733578
610    N49685  0.732717
14603  N46772  0.715329
297    N53585  0.712275
2246   N55943  0.655465
6518   N35645  0.652391


In [20]:
def recommend_similar_content(target_news_id=None, num_recommendations=5):
    if target_news_id is None:
        target_news_id = random.choice(news['News ID'].tolist())
    
    target_embedding = news.loc[news['News ID'] == target_news_id, 'Combined_embedding'].values[0]
    all_news_embeddings = np.vstack(news['Combined_embedding'].values)

    similarity_scores = cosine_similarity([target_embedding], all_news_embeddings).flatten()
    top_indices = np.argsort(similarity_scores)[-num_recommendations-1:-1][::-1]
    return target_news_id, news.iloc[top_indices][['News ID', 'Title', 'URL']]

random_news_id, similar_articles = recommend_similar_content()
print(f"Content Recommendations for News ID {random_news_id}:")
print(similar_articles)

Content Recommendations for News ID N3016:
      News ID                                              Title  \
16366  N48682  senate republicans divided over whether whistl...   
33311  N13533   schiff, gop tangle over witnesses, whistleblower   
45184  N21333  some house gop members call for hunter biden, ...   
9371   N33639     president trump calls out 'fake whistleblower'   
34723  N29696  sen. graham: impeachment 'dead on arrival' in ...   

                                                 URL  
16366  https://assets.msn.com/labs/mind/AAJNvhG.html  
33311  https://assets.msn.com/labs/mind/BBWHGIN.html  
45184  https://assets.msn.com/labs/mind/BBWw7YS.html  
9371   https://assets.msn.com/labs/mind/BBWM9Ga.html  
34723  https://assets.msn.com/labs/mind/BBWyjSP.html  


In [23]:
def generate_hybrid_recommendations(user_identifier=None, target_news_id=None, num_recommendations=5):

    if user_identifier is None:
        user_identifier = random.choice(list(user_id_mapping.keys()))
    if target_news_id is None:
        target_news_id = random.choice(news['News ID'].tolist())

    _, collaborative_recs = generate_recommendations(user_identifier, num_recommendations)
    _, content_recs = recommend_similar_content(target_news_id, num_recommendations)

    collab_news_ids = collaborative_recs[['News ID']]
    content_news_ids = content_recs[['News ID']]

    merged_recommendations = pd.concat(
        [collab_news_ids, content_news_ids],
        ignore_index=True
    ).drop_duplicates()

    return user_identifier, target_news_id, merged_recommendations

random_user_id, random_news_id, hybrid_recommendations = generate_hybrid_recommendations()
print(f"\nHybrid Recommendations for User {random_user_id} and News ID {random_news_id}:")
print(hybrid_recommendations)

634/634 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step

Hybrid Recommendations for User U16982 and News ID N44799:
  News ID
0  N25672
1  N61429
2  N62296
3   N8425
4  N27945
5  N33883
6  N63534
7  N45631
8  N27901
9  N57945
